# TDGraph-IDS -- Phase 4: Machine Learning Classification and SHAP Explainability

**Goal:** Train and evaluate ML models across three feature sets to isolate the
contribution of topology features, then explain predictions with SHAP.

**Inputs:**
- `data/processed/X_train.csv` -- 38-feature SMOTE-balanced training matrix
- `data/processed/X_test.csv`  -- 38-feature test matrix (real distribution)
- `data/processed/enriched_flows.csv` -- flows with 9 topology features

**Outputs:**
- `models/model_rf_flow.pkl`         -- Random Forest (flow only)
- `models/model_xgb_flow.pkl`        -- XGBoost (flow only)
- `models/model_lgbm_flow.pkl`       -- LightGBM (flow only)
- `models/model_ensemble.pkl`        -- Soft voting ensemble (flow only)
- `models/model_xgb_hybrid.pkl`      -- XGBoost (flow + topology)
- `models/model_lgbm_hybrid.pkl`     -- LightGBM (flow + topology)
- `models/best_model.txt`            -- name of best model by F1
- `outputs/reports/model_comparison.csv`
- `outputs/plots/phase4_*.png`

**Experimental design:**
Six models across two groups isolate the topology contribution:

| Group | Models | Features |
|---|---|---|
| Flow-only | RF, XGBoost, LightGBM, Ensemble | 38 (flow + DPI) |
| Hybrid | XGBoost, LightGBM | 38 + 9 topology = 47 |

The F1 and precision difference between groups directly and empirically
quantifies the value added by the dynamic graph component.

---

## 4.1 -- Setup

In [3]:
import sys
import os
import time
import warnings
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
import shap

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import CFG, ensure_dirs

ensure_dirs()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('phase4')

log.info('Project root : %s', CFG.ROOT)
log.info('Models dir   : %s', CFG.MODELS)
log.info('Reports dir  : %s', CFG.REPORTS)

09:54:21  INFO  Project root : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS
09:54:21  INFO  Models dir   : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\models
09:54:21  INFO  Reports dir  : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\reports


## 4.2 -- Load flow-only feature data

Load the SMOTE-balanced training set and real-distribution test set
produced by Phase 2.

In [4]:
def load_flow_data() -> tuple:
    """
    Load the flow-only feature splits from Phase 2.

    Returns
    -------
    tuple of (X_train, X_test, y_train, y_test)
        X_train : pd.DataFrame  SMOTE-balanced training features.
        X_test  : pd.DataFrame  Real-distribution test features.
        y_train : pd.Series     Balanced binary training labels.
        y_test  : pd.Series     Binary test labels.

    Raises
    ------
    FileNotFoundError
        If any required file from Phase 2 is missing.
    """
    required = ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']
    for fname in required:
        path = CFG.DATA_PROCESSED / fname
        if not path.exists():
            raise FileNotFoundError(
                f'{fname} not found. Run Phase 2 first.'
            )

    log.info('Loading flow-only data from Phase 2 ...')

    X_train = pd.read_csv(CFG.DATA_PROCESSED / 'X_train.csv')
    X_test  = pd.read_csv(CFG.DATA_PROCESSED / 'X_test.csv')
    y_train = pd.read_csv(CFG.DATA_PROCESSED / 'y_train.csv').squeeze()
    y_test  = pd.read_csv(CFG.DATA_PROCESSED / 'y_test.csv').squeeze()

    log.info('  X_train : %d rows x %d cols  (SMOTE-balanced)',
             *X_train.shape)
    log.info('  X_test  : %d rows x %d cols  (real distribution)',
             *X_test.shape)
    log.info('  y_train attack rate : %.2f%%', y_train.mean() * 100)
    log.info('  y_test  attack rate : %.2f%%', y_test.mean()  * 100)

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = load_flow_data()

print('X_train shape :', X_train.shape, '(SMOTE-balanced)')
print('X_test  shape :', X_test.shape,  '(real distribution)')
print('y_train attack rate : {:.2f}%'.format(y_train.mean() * 100))
print('y_test  attack rate : {:.2f}%'.format(y_test.mean()  * 100))

09:54:21  INFO  Loading flow-only data from Phase 2 ...
09:54:42  INFO    X_train : 3652252 rows x 38 cols  (SMOTE-balanced)
09:54:42  INFO    X_test  : 473128 rows x 38 cols  (real distribution)
09:54:42  INFO    y_train attack rate : 50.00%
09:54:42  INFO    y_test  attack rate : 3.51%


X_train shape : (3652252, 38) (SMOTE-balanced)
X_test  shape : (473128, 38) (real distribution)
y_train attack rate : 50.00%
y_test  attack rate : 3.51%


## 4.3 -- Build hybrid feature set (flow + topology)

Topology features from Phase 3 are joined onto the flow records using
src_ip as the merge key. Per-node averages across all windows are used.

A fresh StandardScaler is fitted on the topology columns only
(the flow features are already scaled from Phase 2).

Critical: scaler fitted on train split only, then applied to test split.

In [5]:
def build_hybrid_features() -> tuple:
    """
    Build the hybrid feature set by appending topology features to the
    already-scaled X_train and X_test from Phase 2.

    Strategy:
        1. Load topo_node_avg.csv (per-IP average topology features from Phase 3).
        2. Load graph_data.csv to get the src_ip for each flow in order.
        3. Merge topology features onto each flow by src_ip.
        4. Append topology columns to X_train and X_test (which are already
           correctly scaled and split from Phase 2).
        5. Fit a fresh StandardScaler on topology columns using train rows only.

    This avoids re-applying the Phase 2 scaler which was fitted on different
    columns than those present in enriched_flows.csv.

    Returns
    -------
    tuple of (Xh_train, Xh_test, yh_train, yh_test, hybrid_available)
        Xh_train         : pd.DataFrame  Hybrid training features.
        Xh_test          : pd.DataFrame  Hybrid test features.
        yh_train         : pd.Series     Training labels.
        yh_test          : pd.Series     Test labels.
        hybrid_available : bool          False if topology files missing.
    """
    topo_avg_path  = CFG.DATA_GRAPH / 'topo_node_avg.csv'
    graph_data_path = CFG.DATA_GRAPH / 'graph_data.csv'

    if not topo_avg_path.exists():
        log.warning('topo_node_avg.csv not found -- skipping hybrid models')
        return None, None, None, None, False

    if not graph_data_path.exists():
        log.warning('graph_data.csv not found -- skipping hybrid models')
        return None, None, None, None, False

    log.info('Loading topology node averages ...')
    topo_avg = pd.read_csv(topo_avg_path)
    log.info('  topo_node_avg shape: %d rows x %d cols', *topo_avg.shape)

    # Topology feature columns (everything except node identifier)
    topo_feat_cols = [c for c in topo_avg.columns if c != 'node']
    log.info('  Topology features per node: %d', len(topo_feat_cols))

    # Load graph_data to get src_ip in original row order
    log.info('Loading graph_data for IP lookup ...')
    graph_data = pd.read_csv(
        graph_data_path,
        usecols=['src_ip'],
        low_memory=False,
    )

    # Merge topology onto each flow by src_ip
    topo_src = topo_avg.rename(columns=
        {'node': 'src_ip', **{c: f'src_topo_{c}' for c in topo_feat_cols}}
    )
    src_topo_cols = [f'src_topo_{c}' for c in topo_feat_cols]

    merged = graph_data.merge(topo_src, on='src_ip', how='left')
    merged[src_topo_cols] = merged[src_topo_cols].fillna(0)

    # Align topology rows to Phase 2 train/test split
    # Phase 2 used stratified split with same random_state and test_size
    # Reproduce the same split indices on the full 2.3M row sequence
    from sklearn.model_selection import train_test_split as tts

    y_full_graph = pd.read_csv(
        graph_data_path.parent.parent / 'processed' / 'y_test.csv'
    )  # just to get length -- not used directly

    n_total    = len(merged)
    all_idx    = np.arange(n_total)
    y_proxy    = np.zeros(n_total)  # labels not needed for index split

    # Reproduce Phase 2 split indices
    train_idx, test_idx = tts(
        all_idx,
        test_size=CFG.TEST_SIZE,
        random_state=CFG.RANDOM_STATE,
    )

    topo_train = merged[src_topo_cols].iloc[train_idx].reset_index(drop=True)
    topo_test  = merged[src_topo_cols].iloc[test_idx].reset_index(drop=True)

    # Align lengths to X_train and X_test
    # X_train has SMOTE rows -- topology appended to non-SMOTE portion
    # Use X_test rows directly (no SMOTE on test)
    n_test  = len(X_test)
    n_train_orig = len(train_idx)

    # For SMOTE-balanced X_train: append topology to original rows,
    # then fill synthetic rows with mean topology
    if len(topo_train) < len(X_train):
        # X_train has SMOTE synthetic rows -- pad with mean of topology train
        topo_mean = topo_train.mean()
        n_extra   = len(X_train) - len(topo_train)
        extra_rows = pd.DataFrame(
            [topo_mean.values] * n_extra,
            columns=src_topo_cols,
        )
        topo_train_full = pd.concat(
            [topo_train, extra_rows], ignore_index=True
        )
    else:
        topo_train_full = topo_train.iloc[:len(X_train)].reset_index(drop=True)

    topo_test_aligned = topo_test.iloc[:n_test].reset_index(drop=True)

    # Scale topology columns -- fit on train topology only
    topo_scaler = StandardScaler()
    topo_train_sc = pd.DataFrame(
        topo_scaler.fit_transform(topo_train_full),
        columns=src_topo_cols,
    )
    topo_test_sc = pd.DataFrame(
        topo_scaler.transform(topo_test_aligned),
        columns=src_topo_cols,
    )

    # Concatenate with existing flow features
    Xh_train = pd.concat(
        [X_train.reset_index(drop=True), topo_train_sc],
        axis=1,
    )
    Xh_test = pd.concat(
        [X_test.reset_index(drop=True), topo_test_sc],
        axis=1,
    )

    yh_train = y_train.reset_index(drop=True)
    yh_test  = y_test.reset_index(drop=True)

    log.info(
        'Hybrid features: %d total (%d flow + %d topology)',
        Xh_train.shape[1], len(X_train.columns), len(src_topo_cols),
    )
    log.info('Xh_train: %d rows, Xh_test: %d rows', len(Xh_train), len(Xh_test))

    return Xh_train, Xh_test, yh_train, yh_test, True


Xh_train, Xh_test, yh_train, yh_test, HYBRID_AVAILABLE = build_hybrid_features()

if HYBRID_AVAILABLE:
    print('Hybrid feature set ready')
    print('  Xh_train shape :', Xh_train.shape)
    print('  Xh_test  shape :', Xh_test.shape)
    print('  yh_train attack rate : {:.2f}%'.format(yh_train.mean() * 100))
    print('  yh_test  attack rate : {:.2f}%'.format(yh_test.mean()  * 100))
else:
    print('Hybrid features not available -- flow-only models will run')

09:54:42  INFO  Loading topology node averages ...
09:54:42  INFO    topo_node_avg shape: 42 rows x 11 cols
09:54:42  INFO    Topology features per node: 10
09:54:42  INFO  Loading graph_data for IP lookup ...
09:54:55  INFO  Hybrid features: 48 total (38 flow + 10 topology)
09:54:55  INFO  Xh_train: 3652252 rows, Xh_test: 473128 rows


Hybrid feature set ready
  Xh_train shape : (3652252, 48)
  Xh_test  shape : (473128, 48)
  yh_train attack rate : 50.00%
  yh_test  attack rate : 3.51%


## 4.4 -- Evaluation helper

Single function to train, predict, and compute all metrics.
Results are stored in a dict for comparison table construction.

In [6]:
results = {}


def evaluate(
    name: str,
    model,
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_te: pd.DataFrame,
    y_te: pd.Series,
) -> tuple:
    """
    Train a model and compute all evaluation metrics.

    Stores results in the global results dict for comparison.

    Parameters
    ----------
    name  : str          Model name for reporting.
    model : estimator    Unfitted sklearn-compatible model.
    X_tr  : pd.DataFrame Training features.
    y_tr  : pd.Series    Training labels.
    X_te  : pd.DataFrame Test features.
    y_te  : pd.Series    Test labels.

    Returns
    -------
    tuple of (fitted_model, y_pred, y_proba)
    """
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_te,  y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te,    y_pred, zero_division=0)
    f1   = f1_score(y_te,        y_pred, zero_division=0)
    auc  = roc_auc_score(y_te,   y_proba)

    results[name] = {
        'Accuracy' : round(acc,  4),
        'Precision': round(prec, 4),
        'Recall'   : round(rec,  4),
        'F1'       : round(f1,   4),
        'ROC_AUC'  : round(auc,  4),
        'Train_sec': round(train_time, 1),
    }

    log.info(
        '  %-30s  Acc=%.4f  Prec=%.4f  Rec=%.4f  F1=%.4f  AUC=%.4f  t=%.1fs',
        name, acc, prec, rec, f1, auc, train_time,
    )

    return model, y_pred, y_proba


print('Evaluation function ready')

Evaluation function ready


## 4.5 -- Flow-only models (baseline group)

Four models trained on 38 flow + DPI features.
These establish the performance ceiling achievable without topology.
The report shows all three individual models converge within 0.002 F1
of each other, indicating the 38-feature representation has reached
its information ceiling on this data.

In [7]:
print('=' * 70)
print('FLOW-ONLY MODELS (38 features)')
print('=' * 70)

# Random Forest
log.info('Training Random Forest (flow only) ...')
rf = RandomForestClassifier(**CFG.RF_PARAMS)
rf, rf_pred, rf_proba = evaluate(
    'RF -- Flow Only', rf, X_train, y_train, X_test, y_test
)
joblib.dump(rf, CFG.MODELS / 'model_rf_flow.pkl')
log.info('  Saved: model_rf_flow.pkl')

# XGBoost
log.info('Training XGBoost (flow only) ...')
scale_pos = int((y_train == 0).sum() / max((y_train == 1).sum(), 1))
xgb_params = {**CFG.XGB_PARAMS, 'scale_pos_weight': scale_pos}
xgb = XGBClassifier(**xgb_params)
xgb, xgb_pred, xgb_proba = evaluate(
    'XGB -- Flow Only', xgb, X_train, y_train, X_test, y_test
)
joblib.dump(xgb, CFG.MODELS / 'model_xgb_flow.pkl')
log.info('  Saved: model_xgb_flow.pkl')

# LightGBM
log.info('Training LightGBM (flow only) ...')
lgbm = LGBMClassifier(**CFG.LGBM_PARAMS)
lgbm, lgbm_pred, lgbm_proba = evaluate(
    'LightGBM -- Flow Only', lgbm, X_train, y_train, X_test, y_test
)
joblib.dump(lgbm, CFG.MODELS / 'model_lgbm_flow.pkl')
log.info('  Saved: model_lgbm_flow.pkl')

# Soft voting ensemble
log.info('Training Ensemble (soft vote) ...')
ensemble = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb), ('lgbm', lgbm)],
    voting='soft',
)
ensemble, ens_pred, ens_proba = evaluate(
    'Ensemble (Soft Vote)', ensemble, X_train, y_train, X_test, y_test
)
joblib.dump(ensemble, CFG.MODELS / 'model_ensemble.pkl')
log.info('  Saved: model_ensemble.pkl')

print()
print('Flow-only models complete')

09:54:55  INFO  Training Random Forest (flow only) ...



FLOW-ONLY MODELS (38 features)


09:58:44  INFO    RF -- Flow Only                 Acc=0.9966  Prec=0.9112  Rec=0.9992  F1=0.9532  AUC=0.9998  t=226.1s
09:58:44  INFO    Saved: model_rf_flow.pkl
09:58:44  INFO  Training XGBoost (flow only) ...
09:59:06  INFO    XGB -- Flow Only                Acc=0.9966  Prec=0.9133  Rec=0.9978  F1=0.9537  AUC=0.9998  t=21.4s
09:59:06  INFO    Saved: model_xgb_flow.pkl
09:59:06  INFO  Training LightGBM (flow only) ...
09:59:17  INFO    LightGBM -- Flow Only           Acc=0.9965  Prec=0.9098  Rec=0.9979  F1=0.9518  AUC=0.9997  t=10.2s
09:59:17  INFO    Saved: model_lgbm_flow.pkl
09:59:17  INFO  Training Ensemble (soft vote) ...
10:03:29  INFO    Ensemble (Soft Vote)            Acc=0.9966  Prec=0.9124  Rec=0.9989  F1=0.9537  AUC=0.9998  t=249.9s
10:03:29  INFO    Saved: model_ensemble.pkl



Flow-only models complete


## 4.6 -- Hybrid models (flow + topology)

XGBoost and LightGBM trained on 47 features (38 flow + DPI + 9 topology).
The performance difference vs flow-only directly quantifies the value
of the dynamic graph component.

From the report: Hybrid XGBoost achieves F1=0.9695, Precision improves
by +5.25% -- reducing false alarms because topology context distinguishes
legitimate high-bandwidth traffic (stable topology, no degree delta)
from active attack traffic (sudden degree spikes, high out_degree).

In [8]:
if HYBRID_AVAILABLE:
    print('=' * 70)
    print('HYBRID MODELS (flow + DPI + topology)')
    print('=' * 70)

    # XGBoost Hybrid
    log.info('Training XGBoost (hybrid) ...')
    scale_pos_h = int(
        (yh_train == 0).sum() / max((yh_train == 1).sum(), 1)
    )
    xgb_h_params = {**CFG.XGB_PARAMS, 'scale_pos_weight': scale_pos_h}
    xgb_h = XGBClassifier(**xgb_h_params)
    xgb_h, xgb_h_pred, xgb_h_proba = evaluate(
        'XGB -- Hybrid', xgb_h, Xh_train, yh_train, Xh_test, yh_test
    )
    joblib.dump(xgb_h, CFG.MODELS / 'model_xgb_hybrid.pkl')
    log.info('  Saved: model_xgb_hybrid.pkl')

    # LightGBM Hybrid
    log.info('Training LightGBM (hybrid) ...')
    lgbm_h = LGBMClassifier(**CFG.LGBM_PARAMS)
    lgbm_h, lgbm_h_pred, lgbm_h_proba = evaluate(
        'LightGBM -- Hybrid', lgbm_h, Xh_train, yh_train, Xh_test, yh_test
    )
    joblib.dump(lgbm_h, CFG.MODELS / 'model_lgbm_hybrid.pkl')
    log.info('  Saved: model_lgbm_hybrid.pkl')

    print()
    print('Hybrid models complete')
else:
    print('Hybrid models skipped -- enriched_flows.csv not available')

10:03:30  INFO  Training XGBoost (hybrid) ...


HYBRID MODELS (flow + DPI + topology)


10:03:49  INFO    XGB -- Hybrid                   Acc=0.9976  Prec=0.9566  Rec=0.9752  F1=0.9658  AUC=0.9998  t=19.2s
10:03:49  INFO    Saved: model_xgb_hybrid.pkl
10:03:49  INFO  Training LightGBM (hybrid) ...
10:03:59  INFO    LightGBM -- Hybrid              Acc=0.9973  Prec=0.9444  Rec=0.9813  F1=0.9625  AUC=0.9997  t=9.5s
10:03:59  INFO    Saved: model_lgbm_hybrid.pkl



Hybrid models complete


## 4.7 -- Full model comparison table

Print the complete results table and compute the topology feature impact.
This is Table 1 in the paper.

In [9]:
results_df = pd.DataFrame(results).T.reset_index()
results_df = results_df.rename(columns={'index': 'Model'})
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)

print('=' * 80)
print('COMPLETE MODEL COMPARISON TABLE')
print('=' * 80)
print(results_df[['Model','Accuracy','Precision','Recall','F1','ROC_AUC']]
      .to_string(index=False))
print('=' * 80)

# Save comparison report
results_df.to_csv(CFG.REPORTS / 'model_comparison.csv', index=False)
log.info('Saved: model_comparison.csv')

# Find best model
best_row   = results_df.loc[results_df['F1'].idxmax()]
best_name  = best_row['Model']
best_f1    = best_row['F1']

with open(CFG.MODELS / 'best_model.txt', 'w') as f:
    f.write(best_name)

print()
print('Best model : {}'.format(best_name))
print('Best F1    : {:.4f}'.format(best_f1))

# Topology impact analysis
if HYBRID_AVAILABLE:
    flow_only = results_df[
        ~results_df['Model'].str.contains('Hybrid')
    ]
    hybrid = results_df[
        results_df['Model'].str.contains('Hybrid')
    ]

    if len(flow_only) > 0 and len(hybrid) > 0:
        best_flow_f1   = flow_only['F1'].max()
        best_flow_prec = flow_only['Precision'].max()
        best_flow_rec  = flow_only['Recall'].max()
        best_hyb_f1    = hybrid['F1'].max()
        best_hyb_prec  = hybrid['Precision'].max()
        best_hyb_rec   = hybrid['Recall'].max()

        print()
        print('Topology Feature Impact:')
        print('  F1        : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_f1, best_hyb_f1, best_hyb_f1 - best_flow_f1))
        print('  Precision : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_prec, best_hyb_prec, best_hyb_prec - best_flow_prec))
        print('  Recall    : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_rec, best_hyb_rec, best_hyb_rec - best_flow_rec))

COMPLETE MODEL COMPARISON TABLE
                Model  Accuracy  Precision  Recall     F1  ROC_AUC
        XGB -- Hybrid    0.9976     0.9566  0.9752 0.9658   0.9998
   LightGBM -- Hybrid    0.9973     0.9444  0.9813 0.9625   0.9997
 Ensemble (Soft Vote)    0.9966     0.9124  0.9989 0.9537   0.9998
     XGB -- Flow Only    0.9966     0.9133  0.9978 0.9537   0.9998
      RF -- Flow Only    0.9966     0.9112  0.9992 0.9532   0.9998
LightGBM -- Flow Only    0.9965     0.9098  0.9979 0.9518   0.9997


10:04:03  INFO  Saved: model_comparison.csv



Best model : XGB -- Hybrid
Best F1    : 0.9658

Topology Feature Impact:
  F1        : 0.9537 -> 0.9658  (+0.0121)
  Precision : 0.9133 -> 0.9566  (+0.0433)
  Recall    : 0.9992 -> 0.9813  (-0.0179)


## 4.8 -- SHAP Explainability

TreeExplainer computes exact Shapley values for tree models.
Run on the LightGBM flow-only model (fastest SHAP computation)
and the best hybrid model.

From the report: 5 of the top 10 SHAP features are behavioral DPI
features engineered in Phase 2, validating that manually crafted
behavioral signals are among the most globally discriminative predictors.

In [10]:
def compute_shap(
    model,
    X_sample: pd.DataFrame,
    model_name: str,
    sample_size: int = 2000,
) -> tuple:
    """
    Compute SHAP values for a tree model using TreeExplainer.

    Parameters
    ----------
    model      : fitted tree model  LightGBM or XGBoost.
    X_sample   : pd.DataFrame       Test features to explain.
    model_name : str                Name for logging.
    sample_size: int                Number of samples to use.

    Returns
    -------
    tuple of (shap_values, X_shap, explainer)
    """
    log.info('Computing SHAP values for %s ...', model_name)

    X_shap = X_sample.sample(
        n=min(sample_size, len(X_sample)),
        random_state=CFG.RANDOM_STATE,
    )

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    # For binary classification shap_values may be a list [class0, class1]
    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    log.info('  SHAP computed for %d samples, %d features',
             len(X_shap), X_shap.shape[1])
    return sv, X_shap, explainer


# SHAP for LightGBM flow-only
sv_flow, X_shap_flow, _ = compute_shap(
    lgbm, X_test, 'LightGBM -- Flow Only'
)

# SHAP for best hybrid model (if available)
sv_hyb = X_shap_hyb = None
if HYBRID_AVAILABLE:
    best_hybrid = lgbm_h  # LightGBM hybrid is fastest
    sv_hyb, X_shap_hyb, _ = compute_shap(
        best_hybrid, Xh_test, 'LightGBM -- Hybrid'
    )

print('SHAP values computed')

10:04:04  INFO  Computing SHAP values for LightGBM -- Flow Only ...
10:04:04  INFO    SHAP computed for 2000 samples, 38 features
10:04:04  INFO  Computing SHAP values for LightGBM -- Hybrid ...
10:04:04  INFO    SHAP computed for 2000 samples, 48 features


SHAP values computed


## 4.9 -- Adaptive EWMA threshold

Rather than a fixed 0.5 threshold, an EWMA adaptive threshold adjusts
to current traffic conditions:

- During benign periods: threshold is sensitive, capturing subtle anomalies
- During attack bursts: threshold rises to reduce false alarms
- Alpha=0.05 ensures slow adaptation (attacker cannot manipulate by
  sending brief benign traffic immediately before an attack)

In [11]:
def compute_ewma_threshold(
    scores: np.ndarray,
    alpha: float = 0.05,
) -> tuple:
    """
    Compute EWMA adaptive detection threshold over a score sequence.

    Parameters
    ----------
    scores : np.ndarray  Anomaly score sequence (0-1).
    alpha  : float       EWMA smoothing factor (report specifies 0.05).

    Returns
    -------
    tuple of (ewma, ewma_std, threshold)
        ewma      : np.ndarray  EWMA of scores.
        ewma_std  : np.ndarray  EWMA standard deviation.
        threshold : np.ndarray  Adaptive threshold = ewma + 2*ewma_std.
    """
    ewma     = np.zeros(len(scores))
    ewma_std = np.zeros(len(scores))
    ewma[0]     = scores[0]
    ewma_std[0] = 0.0

    for i in range(1, len(scores)):
        ewma[i]     = alpha * scores[i] + (1 - alpha) * ewma[i-1]
        ewma_std[i] = np.sqrt(
            alpha * (scores[i] - ewma[i])**2 +
            (1 - alpha) * ewma_std[i-1]**2
        )

    threshold = ewma + 2 * ewma_std
    return ewma, ewma_std, threshold


ewma, ewma_std, threshold = compute_ewma_threshold(lgbm_proba, alpha=0.05)

print('EWMA threshold computed')
print('  Mean threshold : {:.4f}'.format(threshold.mean()))
print('  Min threshold  : {:.4f}'.format(threshold.min()))
print('  Max threshold  : {:.4f}'.format(threshold.max()))

EWMA threshold computed
  Mean threshold : 0.3715
  Min threshold  : 0.0000
  Max threshold  : 1.0989


## 4.10 -- Visualizations for submission

Six plots saved to `outputs/plots/` at 200 DPI:
1. Confusion matrices for all 4+ models
2. ROC curves: flow-only vs hybrid
3. Metric comparison bar chart
4. SHAP summary plot (LightGBM flow-only)
5. SHAP bar chart with DPI vs flow feature coloring
6. Adaptive EWMA threshold visualization

In [12]:
plt.rcParams.update({
    'figure.dpi'       : 200,
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})


def save_figure(fig: plt.Figure, filename: str) -> None:
    """
    Save a figure to the plots directory at 200 DPI and close it.

    Parameters
    ----------
    fig      : plt.Figure  Figure to save.
    filename : str         Output filename.
    """
    path = CFG.PLOTS / filename
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    log.info('Saved: %s', path)


# Plot 1: Confusion matrices
flow_model_preds = [
    ('RF -- Flow Only',       y_test,  rf_pred),
    ('XGB -- Flow Only',      y_test,  xgb_pred),
    ('LightGBM -- Flow Only', y_test,  lgbm_pred),
    ('Ensemble (Soft Vote)',  y_test,  ens_pred),
]
if HYBRID_AVAILABLE:
    flow_model_preds += [
        ('XGB -- Hybrid',      yh_test, xgb_h_pred),
        ('LightGBM -- Hybrid', yh_test, lgbm_h_pred),
    ]

n_models = len(flow_model_preds)
ncols    = 3
nrows    = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(5 * ncols, 4.5 * nrows))
axes = axes.flatten() if n_models > 1 else [axes]
fig.suptitle('Confusion Matrices -- All Models', fontsize=14, fontweight='bold')

for i, (name, y_true, y_pred_i) in enumerate(flow_model_preds):
    cm = confusion_matrix(y_true, y_pred_i)
    sns.heatmap(
        cm, annot=True, fmt=',', cmap='Blues',
        ax=axes[i],
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack'],
        linewidths=0.5,
    )
    f1_i  = f1_score(y_true, y_pred_i, zero_division=0)
    acc_i = accuracy_score(y_true, y_pred_i)
    axes[i].set_title(
        '{}\nF1={:.4f}  Acc={:.4f}'.format(name, f1_i, acc_i),
        fontsize=10,
    )
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(n_models, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save_figure(fig, 'phase4_confusion_matrices.png')


# Plot 2: ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ROC Curves -- Flow-Only vs Hybrid', fontsize=13, fontweight='bold')

flow_roc = [
    ('Random Forest',  rf_proba,   y_test, '#e74c3c'),
    ('XGBoost',        xgb_proba,  y_test, '#2c7bb6'),
    ('LightGBM',       lgbm_proba, y_test, '#27ae60'),
    ('Ensemble',       ens_proba,  y_test, '#8e44ad'),
]
for name, proba, true, col in flow_roc:
    fpr, tpr, _ = roc_curve(true, proba)
    auc_val = roc_auc_score(true, proba)
    axes[0].plot(fpr, tpr, color=col, lw=2,
                 label='{} (AUC={:.4f})'.format(name, auc_val))
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_title('Flow-Only Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

if HYBRID_AVAILABLE:
    hybrid_roc = [
        ('XGBoost Hybrid',  xgb_h_proba,  yh_test, '#d35400'),
        ('LightGBM Hybrid', lgbm_h_proba, yh_test, '#16a085'),
    ]
    for name, proba, true, col in hybrid_roc:
        fpr, tpr, _ = roc_curve(true, proba)
        auc_val = roc_auc_score(true, proba)
        axes[1].plot(fpr, tpr, color=col, lw=2,
                     label='{} (AUC={:.4f})'.format(name, auc_val))
    axes[1].plot([0,1],[0,1],'k--',lw=1)
    axes[1].set_title('Hybrid Models (Flow + Topology)')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].legend(loc='lower right', fontsize=9)
    axes[1].grid(alpha=0.3)
else:
    axes[1].set_visible(False)

save_figure(fig, 'phase4_roc_curves.png')


# Plot 3: Metric comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
comp_df = results_df.set_index('Model')[metrics_to_plot].apply(
    pd.to_numeric, errors='coerce'
)

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')

palette = ['#e74c3c','#2c7bb6','#27ae60','#8e44ad','#d35400','#16a085']

for i, metric in enumerate(metrics_to_plot):
    vals   = comp_df[metric].values
    labels = comp_df.index.tolist()
    colors = palette[:len(vals)]
    bars   = axes[i].bar(range(len(vals)), vals,
                          color=colors, edgecolor='white', linewidth=0.8)
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_ylim(
        max(0, float(np.nanmin(vals)) - 0.02),
        min(1.0, float(np.nanmax(vals)) + 0.02)
    )
    axes[i].set_xticks(range(len(vals)))
    axes[i].set_xticklabels(
        [m.split('--')[-1].strip() for m in labels],
        rotation=35, ha='right', fontsize=8,
    )
    for bar, val in zip(bars, vals):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            '{:.3f}'.format(val),
            ha='center', fontsize=7.5,
        )
    # Gold border on best bar
    best_idx = int(np.nanargmax(vals))
    bars[best_idx].set_edgecolor('goldenrod')
    bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
save_figure(fig, 'phase4_metric_comparison.png')


# Plot 4: SHAP summary (LightGBM flow-only)
DPI_FEATURE_COLS = [
    'byte_entropy', 'pkt_size_mean', 'pkt_size_ratio',
    'iat_variance', 'iat_cv', 'burst_score',
    'flow_efficiency', 'direction_asymmetry',
    'protocol_anomaly', 'syn_ratio', 'rst_ratio',
    'is_ephemeral_src', 'is_well_known_dst', 'is_suspicious_port',
    'flow_bytes_per_sec', 'flow_pkts_per_sec',
]

mean_shap = np.abs(sv_flow).mean(axis=0)
shap_df   = pd.DataFrame({
    'feature'   : X_shap_flow.columns,
    'importance': mean_shap,
})
shap_df   = shap_df.nlargest(20, 'importance').sort_values('importance')

bar_colors = [
    '#d7191c' if f in DPI_FEATURE_COLS else '#2c7bb6'
    for f in shap_df['feature']
]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(shap_df['feature'], shap_df['importance'],
        color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_title(
    'Top 20 Features by SHAP Impact -- LightGBM (Flow Only)\n'
    'Red = Behavioral DPI feature  |  Blue = Raw flow feature',
    fontsize=11, fontweight='bold',
)
ax.set_xlabel('Mean |SHAP value|')

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#d7191c', label='Behavioral DPI feature'),
    mpatches.Patch(color='#2c7bb6', label='Raw flow feature'),
], loc='lower right', fontsize=9)
save_figure(fig, 'phase4_shap_bar_flow.png')


# Plot 5: SHAP hybrid (if available)
if HYBRID_AVAILABLE and sv_hyb is not None:
    mean_shap_h = np.abs(sv_hyb).mean(axis=0)
    shap_df_h   = pd.DataFrame({
        'feature'   : X_shap_hyb.columns,
        'importance': mean_shap_h,
    })
    shap_df_h = shap_df_h.nlargest(20, 'importance').sort_values('importance')

    TOPO_COLS_SET = set(
        c for c in X_shap_hyb.columns
        if c.startswith('src_topo_') or c.startswith('dst_topo_')
    )
    bar_colors_h = [
        '#27ae60' if f in TOPO_COLS_SET else
        '#d7191c' if f in DPI_FEATURE_COLS else
        '#2c7bb6'
        for f in shap_df_h['feature']
    ]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(shap_df_h['feature'], shap_df_h['importance'],
            color=bar_colors_h, edgecolor='white', linewidth=0.5)
    ax.set_title(
        'Top 20 Features by SHAP Impact -- LightGBM (Hybrid)\n'
        'Green = Topology  |  Red = DPI  |  Blue = Flow',
        fontsize=11, fontweight='bold',
    )
    ax.set_xlabel('Mean |SHAP value|')
    ax.legend(handles=[
        mpatches.Patch(color='#27ae60', label='Topology feature'),
        mpatches.Patch(color='#d7191c', label='DPI feature'),
        mpatches.Patch(color='#2c7bb6', label='Flow feature'),
    ], loc='lower right', fontsize=9)
    save_figure(fig, 'phase4_shap_bar_hybrid.png')


# Plot 6: EWMA adaptive threshold
N = min(5000, len(lgbm_proba))
x = np.arange(N)
y_arr   = y_test.values[:N]

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(x[y_arr==0], lgbm_proba[:N][y_arr==0],
           s=1, color='#2c7bb6', alpha=0.3, label='Benign')
ax.scatter(x[y_arr==1], lgbm_proba[:N][y_arr==1],
           s=4, color='#d7191c', alpha=0.6, label='Attack')
ax.plot(x, threshold[:N], color='#f4a11d', lw=2,
        label='EWMA adaptive threshold (alpha=0.05)')
ax.fill_between(x, ewma[:N] - ewma_std[:N], ewma[:N] + ewma_std[:N],
                alpha=0.12, color='#f4a11d')
ax.set_title('Adaptive EWMA Threshold -- LightGBM Anomaly Scores',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Flow index')
ax.set_ylabel('Anomaly score')
ax.legend(markerscale=5, fontsize=9)
save_figure(fig, 'phase4_ewma_threshold.png')

print('All plots saved to outputs/plots/')

10:04:14  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_confusion_matrices.png
10:04:16  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_roc_curves.png
10:04:17  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_metric_comparison.png
10:04:17  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_shap_bar_flow.png
10:04:17  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_shap_bar_hybrid.png
10:04:18  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase4_ewma_threshold.png


All plots saved to outputs/plots/


## 4.11 -- Final summary

Print the complete end-to-end summary for the paper results section.

In [13]:
print('=' * 70)
print('PHASE 4 -- FINAL SUMMARY')
print('=' * 70)

print('Model comparison:')
print(results_df[['Model','Accuracy','Precision','Recall','F1','ROC_AUC']]
      .to_string(index=False))
print()

if HYBRID_AVAILABLE:
    flow_only_models = [
        k for k in results if 'Hybrid' not in k
    ]
    hybrid_models = [
        k for k in results if 'Hybrid' in k
    ]

    if flow_only_models and hybrid_models:
        best_flow_f1   = max(results[k]['F1']        for k in flow_only_models)
        best_flow_prec = max(results[k]['Precision']  for k in flow_only_models)
        best_flow_rec  = max(results[k]['Recall']     for k in flow_only_models)
        best_hyb_f1    = max(results[k]['F1']        for k in hybrid_models)
        best_hyb_prec  = max(results[k]['Precision']  for k in hybrid_models)
        best_hyb_rec   = max(results[k]['Recall']     for k in hybrid_models)

        print('Topology feature impact:')
        print('  F1        : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_f1,   best_hyb_f1,   best_hyb_f1   - best_flow_f1))
        print('  Precision : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_prec, best_hyb_prec, best_hyb_prec - best_flow_prec))
        print('  Recall    : {:.4f} -> {:.4f}  ({:+.4f})'.format(
            best_flow_rec,  best_hyb_rec,  best_hyb_rec  - best_flow_rec))
        print()

print('Outputs saved:')
for fname in ['model_rf_flow.pkl', 'model_xgb_flow.pkl',
              'model_lgbm_flow.pkl', 'model_ensemble.pkl',
              'model_xgb_hybrid.pkl', 'model_lgbm_hybrid.pkl']:
    p = CFG.MODELS / fname
    if p.exists():
        print('  {:<35} {:.2f} MB'.format(fname, p.stat().st_size / 1e6))

print('=' * 70)
print('PHASE 4 COMPLETE')
print('=' * 70)
print()
print('Next: run Phase5_Evaluation_Report.ipynb')

PHASE 4 -- FINAL SUMMARY
Model comparison:
                Model  Accuracy  Precision  Recall     F1  ROC_AUC
        XGB -- Hybrid    0.9976     0.9566  0.9752 0.9658   0.9998
   LightGBM -- Hybrid    0.9973     0.9444  0.9813 0.9625   0.9997
 Ensemble (Soft Vote)    0.9966     0.9124  0.9989 0.9537   0.9998
     XGB -- Flow Only    0.9966     0.9133  0.9978 0.9537   0.9998
      RF -- Flow Only    0.9966     0.9112  0.9992 0.9532   0.9998
LightGBM -- Flow Only    0.9965     0.9098  0.9979 0.9518   0.9997

Topology feature impact:
  F1        : 0.9537 -> 0.9658  (+0.0121)
  Precision : 0.9133 -> 0.9566  (+0.0433)
  Recall    : 0.9992 -> 0.9813  (-0.0179)

Outputs saved:
  model_rf_flow.pkl                   78.13 MB
  model_xgb_flow.pkl                  1.42 MB
  model_lgbm_flow.pkl                 0.71 MB
  model_ensemble.pkl                  160.52 MB
  model_xgb_hybrid.pkl                1.31 MB
  model_lgbm_hybrid.pkl               0.71 MB
PHASE 4 COMPLETE

Next: run Phase5_Evalua